In [ ]:
import sys, os, warnings
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import Markdown, display
import config
from src.utils import set_seed
set_seed(config.SEED)
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.facecolor": "#F5F1E8", "figure.facecolor": "#F5F1E8", "axes.edgecolor": "#77766D",
                     "font.size": 10, "axes.titleweight": "bold"})
PALETTE = ["#344C3D", "#BD755A", "#C4A46A", "#899982", "#77766D", "#242723"]
sns.set_palette(PALETTE)
def note(txt):  # data-driven observation rendered as markdown
    display(Markdown("> **Observation.** " + txt))

# 09 — Explainability and Error Analysis

Opens the black box: *where* do the models fail, *why*, and how far can their scores be trusted?

Data: the saved per-sentence probabilities on the **shared 2,000-sentence test subset** (`artifacts/evaluation/preds_*.npz`, written by notebooks 04–06), so all models are analysed on identical rows. Models that were not trained (missing artifacts) are skipped — nothing is simulated.

> **Reading guide.** SHAP/LIME/occlusion explain *what the model used*; they do **not** show medical causation, and for transformers they approximate behaviour through perturbations. Attention weights are deliberately *not* used as explanations (see notebook 06).

In [ ]:
import glob, joblib
import src.data as D
from src.utils import load_json, save_json
from src.evaluation import classification_metrics, expected_calibration_error
from src.classification import available_models, get_classifier
LAB = config.CLASS_LABELS; L2I = {l: i for i, l in enumerate(LAB)}
test = D.load_pubmed_rct("test"); eval_sub = D.eval_subset(test).reset_index(drop=True)
M = load_json(config.ART_CLS / "metrics.json", {})
preds = {p.split("preds_")[1][:-4]: np.load(p) for p in glob.glob(str(config.ART_EVAL / "preds_*.npz"))}
names = {k: M[k]["label"] for k in preds if k in M}
print("models with saved predictions:", list(names))
y = eval_sub.label.map(L2I).values
assert all((preds[k]["y"] == y).all() for k in names), "prediction files are not aligned with the shared subset"
best = max(names, key=lambda k: M[k]["f1_macro"]); print("best on shared subset:", best, round(M[best]["f1_macro"], 4))

## 1. Confusion matrices and error types

In [ ]:
show = [k for k in ["tfidf_lr", "tfidf_svm", "bilstm", "distilbert", "bert", "biobert"] if k in names]
fig, axes = plt.subplots(1, len(show), figsize=(3.8 * len(show), 3.8))
axes = np.atleast_1d(axes)
for ax, k in zip(axes, show):
    cm = np.array(M[k]["confusion_matrix"]); sns.heatmap(cm, annot=True, fmt="d", cmap="YlOrBr", xticklabels=[l[:4] for l in LAB], yticklabels=[l[:4] for l in LAB], cbar=False, ax=ax); ax.set_title(names[k], fontsize=9); ax.set_xlabel("predicted")
plt.tight_layout(); plt.show()
cm = np.array(M[best]["confusion_matrix"]); off = sorted([(LAB[i], LAB[j], cm[i, j], cm[i, j] / cm[i].sum()) for i in range(5) for j in range(5) if i != j], key=lambda t: -t[2])
note(f"Best model ({names[best]}) — most frequent confusions on the {len(eval_sub):,}-sentence subset: " + "; ".join(f"**{a}→{b}** ×{n} ({r:.0%} of true {a})" for a, b, n, r in off[:4]) + ".")

### 1.1 False positives and false negatives per class

For class *c*: **false negatives** = sentences truly *c* predicted as something else; **false positives** = sentences predicted *c* that are not.

In [ ]:
P = preds[best]["probs"]; pred = P.argmax(1)
fnfp = pd.DataFrame({LAB[c]: {"false negatives": int(((y == c) & (pred != c)).sum()), "false positives": int(((y != c) & (pred == c)).sum()), "true positives": int(((y == c) & (pred == c)).sum())} for c in range(5)}).T
display(fnfp); fnfp[["false negatives", "false positives"]].plot.bar(figsize=(7, 3), color=[PALETTE[1], PALETTE[3]]); plt.title(f"Error counts per class — {names[best]}"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

### 1.2 Misclassified samples

In [ ]:
mis = eval_sub.assign(pred=[LAB[i] for i in pred], conf=P.max(1)).query("label != pred")
display(mis.sort_values("conf", ascending=False).head(12)[["label", "pred", "conf", "text"]].round(3))
note(f"{len(mis)} of {len(eval_sub)} sentences are misclassified ({len(mis)/len(eval_sub):.1%}). The table lists the *most confident* mistakes: many are sentences that are genuinely ambiguous in isolation "
     f"— read them to judge whether they are ambiguous without context (hypothesis: label noise and missing context explain many of them, consistent with the conflicting-label count of notebook 01; not tested here).")

## 2. Error patterns

Error rate as a function of sentence length, position in the abstract and class — checks whether failures are systematic.

In [ ]:
df = eval_sub.assign(correct=(pred == y), n_words=eval_sub.text.str.split().str.len(), rel_pos=eval_sub.sent_idx / (eval_sub.n_sents - 1).clip(lower=1))
df["len_bin"] = pd.qcut(df.n_words, 5, duplicates="drop"); df["pos_bin"] = pd.cut(df.rel_pos, [-.01, .2, .4, .6, .8, 1.0], labels=["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"])
fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
(1 - df.groupby("len_bin", observed=True).correct.mean()).plot.bar(ax=axes[0], color=PALETTE[0]); axes[0].set_title("Error rate by sentence length (quintiles, words)"); axes[0].tick_params(axis="x", rotation=30)
(1 - df.groupby("pos_bin", observed=True).correct.mean()).plot.bar(ax=axes[1], color=PALETTE[1]); axes[1].set_title("Error rate by position in abstract"); axes[1].tick_params(axis="x", rotation=0)
(1 - df.groupby("label").correct.mean()).reindex(LAB).plot.bar(ax=axes[2], color=PALETTE[2]); axes[2].set_title("Error rate by true class"); axes[2].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()
pe = 1 - df.groupby("pos_bin", observed=True).correct.mean(); le = 1 - df.groupby("len_bin", observed=True).correct.mean()
note(f"Error is highest for the **{pe.idxmax()}** position bin ({pe.max():.1%}) and lowest for **{pe.idxmin()}** ({pe.min():.1%}); across sentence-length quintiles it ranges {le.min():.1%}–{le.max():.1%}. "
     f"A plausible (untested) explanation is that role transitions cluster in some positions, where a sentence-only model has no positional cue — see notebook 04 for the position ablation.")

## 3. Confidence distribution and calibration

Scores are **softmax probabilities / SVM margins, not guaranteed probabilities**. A model is *calibrated* if, among predictions made with confidence ≈ p, a fraction p are correct. **Expected Calibration Error (ECE)** averages |accuracy − confidence| over 10 confidence bins.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4)); ece_rows = {}
for k in names:
    r = expected_calibration_error(preds[k]["probs"], y); ece_rows[k] = {"model": names[k], "ECE": r["ece"], "accuracy": float((preds[k]["probs"].argmax(1) == y).mean()), "mean confidence": float(preds[k]["probs"].max(1).mean())}
    b = pd.DataFrame(r["bins"]); axes[0].plot(b.confidence, b.accuracy, "o-", label=f"{names[k]} (ECE {r['ece']:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=1); axes[0].set_xlabel("mean confidence in bin"); axes[0].set_ylabel("accuracy in bin"); axes[0].set_title("Reliability diagram"); axes[0].legend(fontsize=7)
c = preds[best]["probs"].max(1); ok = pred == y
axes[1].hist([c[ok], c[~ok]], bins=20, stacked=True, color=[PALETTE[0], PALETTE[1]], label=["correct", "wrong"]); axes[1].set_title(f"Confidence histogram — {names[best]}"); axes[1].legend(); axes[1].set_xlabel("max class score")
plt.tight_layout(); plt.show(); ece = pd.DataFrame(ece_rows).T.set_index("model"); display(ece.round(4)); save_json({k: v for k, v in ece_rows.items()}, config.ART_EVAL / "calibration.json")
over = ece[ece["mean confidence"] > ece["accuracy"]].index.tolist()
note(f"Mean confidence exceeds accuracy (over-confidence) for: {over if over else 'none'}. Highest ECE: **{ece['ECE'].idxmax()}** ({ece['ECE'].max():.3f}); lowest: **{ece['ECE'].idxmin()}** ({ece['ECE'].min():.3f}). "
     f"Wrong predictions of {names[best]} have mean confidence {c[~ok].mean():.2f} vs {c[ok].mean():.2f} for correct ones — confidence is informative but wrong answers are rarely *low*-confidence, hence the UI's explicit 'uncalibrated score' label.")

## 4. Feature importance and SHAP (Logistic Regression)

For a linear model with TF-IDF features, SHAP values have a closed form: `coef_c · (x − E[x])` per feature. We use `shap.LinearExplainer` (independent-features assumption) on a 300-sentence background drawn from the **training** set.

In [ ]:
import shap
lr = joblib.load(config.ART_CLS / "models" / "tfidf_lr.joblib"); vec, clf = lr.steps[0][1], lr.steps[-1][1]
train = D.load_pubmed_rct("train"); bg = vec.transform(train.text.sample(300, random_state=0)); Xs = vec.transform(eval_sub.text.head(200))
classes = list(clf.classes_); names_f = vec.get_feature_names_out()
try:
    ex = shap.LinearExplainer(clf, bg); sv = ex.shap_values(Xs); sv = np.stack(sv, -1) if isinstance(sv, list) else np.asarray(sv); how = "shap.LinearExplainer"
except Exception as e:
    mu = np.asarray(bg.mean(0)).ravel(); sv = np.stack([np.asarray(Xs.multiply(1).toarray() - mu) * clf.coef_[c] for c in range(5)], -1); how = f"closed-form fallback ({type(e).__name__})"
print("SHAP computed with:", how, "| shape", sv.shape)
mean_abs = np.abs(sv).mean(0)   # (features, classes)
top = {l: [names_f[i] for i in np.argsort(-mean_abs[:, classes.index(l)])[:8]] for l in LAB}
display(pd.DataFrame(top, index=range(1, 9)))

In [ ]:
i = 3; lab_true = eval_sub.label.iloc[i]; c = classes.index(lr.predict([eval_sub.text.iloc[i]])[0]); row = sv[i, :, c]; nz = np.argsort(-np.abs(row))[:10]
fig, ax = plt.subplots(figsize=(7, 3.6)); ax.barh([names_f[j] for j in nz][::-1], row[nz][::-1], color=[PALETTE[0] if v > 0 else PALETTE[1] for v in row[nz][::-1]]); ax.set_title(f"SHAP for one sentence → predicted {classes[c]}"); plt.tight_layout(); plt.show()
print("sentence:", eval_sub.text.iloc[i], "| true:", lab_true)
note("Positive bars push the prediction *toward* the predicted class, negative bars away. The global table above lists, per class, the features with the largest mean |SHAP|; "
     "inspect whether they are semantically sensible (e.g. statistical terms for RESULTS) or artefacts (identifiers) — the latter would signal shortcut learning.")

## 5. LIME and occlusion for a transformer (or BiLSTM)

LIME fits a sparse linear model on perturbed copies of one sentence (random word removal) to approximate the *local* decision; **occlusion** (used in the web app) removes each word once. Both are slow for transformers, so they are shown for one sentence each and are *local approximations*, not ground truth.

In [ ]:
from lime.lime_text import LimeTextExplainer
from src.explainability import occlusion_terms, linear_terms
tk = next((k for k in ["distilbert", "biobert", "bert", "bilstm"] if k in available_models()), None)
sent = "No significant differences were found between the two treatment groups in mean systolic blood pressure."
if tk:
    cl = get_classifier(tk); pr = cl.predict_proba([sent])[0]; t = int(pr.argmax())
    lime_exp = LimeTextExplainer(class_names=LAB).explain_instance(sent, cl.predict_proba, num_features=8, num_samples=300, labels=(t,))
    occ = occlusion_terms(cl.predict_proba, sent, t, k=8)
    print(f"model: {tk} | predicted {LAB[t]} ({pr[t]:.2f})")
    display(pd.DataFrame({"LIME term": [w for w, _ in lime_exp.as_list(label=t)], "LIME weight": [round(v, 3) for _, v in lime_exp.as_list(label=t)]}))
    display(pd.DataFrame(occ).rename(columns={"term": "occlusion term", "weight": "Δ class score"}).round(3))
else:
    print("No non-classical model artifacts found; train notebooks 05/06 first.")

## 6. Limitations and fairness considerations

* **Sentence in isolation:** the models see no neighbouring sentences; the position analysis shows this causes systematic mid-abstract errors.
* **Label noise:** identical sentences with different labels exist (notebook 01) — an irreducible error floor.
* **Calibration:** scores are uncalibrated; ECE values above quantify it. No calibration layer (Platt/temperature) was fitted — a straightforward future step.
* **Small transformer budget:** transformer errors partly reflect under-training (3,000 sentences), not architecture limits.
* **Fairness:** the dataset contains **no demographic attributes** (age, sex, ethnicity) and abstracts describe study populations only in free text, so *group-fairness metrics cannot be computed*. The only subgroup analysis possible is by sentence length/position (above). Medical-literature corpora also over-represent English-language, high-income-country trials, so behaviour on other literatures is unknown.
* **Attribution ≠ causation:** SHAP/LIME/occlusion describe model behaviour in this dataset, not clinical mechanisms.